# Phase 2.3 — Exploration des données (EDA)

**Projet** : COVID-19 et scolarisation au Sénégal — Panel ménages 2021-2022
**Notebook** : `python/01_eda.ipynb`
**Objectif** : explorer les données préparées par `00_prepare_data.py`, vérifier la qualité, identifier les pièges méthodologiques, et cadrer les questions analytiques pour la suite.

Ce notebook n'est pas le livrable analytique final — c'est une exploration. Le notebook `02_analysis.ipynb` produira les insights définitifs avec contrôle pour l'effet de cohorte et les modèles économétriques.

## Setup

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration affichage
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.3f}'.format)

sns.set_theme(style='whitegrid', context='notebook', palette='deep')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150

# Chemins
ROOT = Path.cwd().parents[0] if Path.cwd().name == 'python' else Path.cwd()
PROC = ROOT / 'data' / 'processed'
print('ROOT :', ROOT)
print('PROC :', PROC)
print('Fichiers disponibles :', [p.name for p in PROC.glob('*')])

In [ ]:
# Chargement des données préparées
menages = pd.read_parquet(PROC / 'menages_panel.parquet')
enfants = pd.read_parquet(PROC / 'enfants_panel.parquet')
dico = pd.read_csv(PROC / 'dictionnaire_variables.csv')

print(f"menages_panel : {menages.shape}")
print(f"enfants_panel : {enfants.shape}")
print(f"dictionnaire  : {dico.shape}")

## 1. Vue d'ensemble du panel

On commence par confirmer la structure : combien de ménages, combien d'enfants, comment se répartissent-ils entre les deux vagues ?

In [ ]:
# Composition du panel ménage
print('=== Panel ménages ===')
print(f"Total lignes : {len(menages)}")
print(f"Ménages uniques (hh_id) : {menages['hh_id'].nunique()}")
print(f"Ménages dans les deux vagues : {(menages['in_panel']).groupby(menages['hh_id']).any().sum()}")
print()
print(pd.crosstab(menages['wave'], menages['in_panel'], margins=True, margins_name='Total'))

In [ ]:
# Composition du panel enfants
print('=== Panel enfants ===')
print(f"Total lignes (enfant × vague) : {len(enfants)}")
print(f"Enfants uniques (hh_id × slot) : {enfants[['hh_id','enfant_slot']].drop_duplicates().shape[0]}")
print()
print('Répartition par statut :')
print(enfants['statut_enfant'].value_counts().to_string())
print()
print('Croisement wave × statut :')
print(pd.crosstab(enfants['wave'], enfants['statut_enfant'], margins=True, margins_name='Total'))

**À retenir** :
- `panel_present` : enfants observés aux deux vagues (cœur de l'analyse panel)
- `only_2021` : enfants disparus en 2022 (ménage perdu ou enfant sorti du ménage)
- `only_2022` : entrées (typiquement données 2021 manquantes pour ce slot)

## 2. Qualité des données

Avant tout calcul de taux, vérifier le taux de complétude des variables clés.

In [ ]:
# Complétude par vague pour les variables analytiques
vars_clefs = ['age_enfant', 'sexe_enfant', 'scolarise', 'classe', 'type_ecole',
              'raison_non_scol', 'cours_particuliers', 'region', 'sexe_cm', 'niveau_educ_cm']

completude = (enfants.groupby('wave')[vars_clefs]
              .apply(lambda d: d.notna().mean() * 100)
              .round(1)
              .T)
completude.columns = ['% non-null 2021', '% non-null 2022']
print('Taux de complétude par variable et vague :')
print(completude.to_string())

**Points d'attention** typiques :
- Si `age_enfant` / `sexe_enfant` ont une complétude < 80 % en 2021, c'est parce que le module démographique enfant (`age_enfant_N`) n'est pas systématiquement rempli — pour certains ménages, seules les questions de scolarisation sont posées sans repasser par la fiche démographique. À considérer pour les analyses qui croisent l'âge avec d'autres variables.
- En 2022, les variables démographiques sont issues du **carry-forward** depuis 2021 — leur complétude reflète directement celle de 2021 pour les enfants du panel.

In [ ]:
# Inspection des doublons résiduels hh_id (signalés par le pipeline)
dup_hh = (menages.groupby(['hh_id','wave'])
          .size()
          .reset_index(name='n')
          .query('n > 1'))
print(f"Doublons (hh_id, wave) après pipeline : {len(dup_hh)}")
if len(dup_hh) > 0:
    print(dup_hh.head(10).to_string())

## 3. Profil démographique des enfants

Qui sont ces enfants dont on suit la scolarité ?

In [ ]:
# Distribution âge × sexe en 2021 (référence démographique)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ref = enfants[enfants['wave'] == 2021].dropna(subset=['age_enfant', 'sexe_enfant'])

sns.histplot(data=ref, x='age_enfant', hue='sexe_enfant', multiple='dodge',
             bins=range(0, 26), ax=axes[0], shrink=0.85)
axes[0].set_title("Distribution des âges (enfants 2021) — n = {}".format(len(ref)))
axes[0].set_xlabel('Âge en années révolues')
axes[0].set_ylabel('Nombre d\'enfants')

# Boîte : âge par sexe
sns.boxplot(data=ref, x='sexe_enfant', y='age_enfant', ax=axes[1], width=0.5)
axes[1].set_title("Âge par sexe")
axes[1].set_xlabel('')
axes[1].set_ylabel('Âge')

plt.tight_layout()
plt.show()

In [ ]:
# Répartition régionale
geo = (enfants[enfants['wave'] == 2021]
       .dropna(subset=['region'])
       .groupby('region')
       .agg(n_menages=('hh_id', 'nunique'),
            n_enfants=('hh_id', 'count'))
       .sort_values('n_menages', ascending=False))
print('Répartition par région (vague 2021) :')
print(geo.to_string())

## 4. Question analytique 1 — Trajectoire de scolarisation

**Hypothèse à tester** : la pandémie a entraîné une baisse de la scolarisation, dont l'ampleur de la reprise est visible entre 2021 et 2022.

⚠️ **Piège méthodologique majeur** : comparer simplement la scolarisation 2021 vs 2022 mélange deux effets :
1. **Effet de cohorte** : entre 2021 et 2022, les enfants vieillissent d'un an. Certains atteignent l'âge où la scolarisation chute naturellement (15-17 ans dans le contexte sénégalais).
2. **Effet COVID/post-COVID** : ce qu'on cherche réellement.

Pour isoler l'effet COVID, on doit **comparer à âge égal** (et idéalement à individu égal — c'est l'avantage du panel).

In [ ]:
# 4a. Vue brute (à âge non contrôlé)
panel = enfants[enfants['statut_enfant'] == 'panel_present'].copy()

taux_brut = (panel.dropna(subset=['scolarise'])
             .groupby('wave')['scolarise']
             .apply(lambda s: (s == 'Oui').mean() * 100)
             .round(1))
print('Taux de scolarisation brut, enfants panel :')
print(taux_brut.to_string())
print(f"\nDifférence 2022 - 2021 : {taux_brut[2022] - taux_brut[2021]:+.1f} points de pourcentage")

In [ ]:
# 4b. À âge contrôlé — par tranche d'âge × vague
panel_age = panel.dropna(subset=['scolarise', 'age_enfant']).copy()
panel_age['groupe_age'] = pd.cut(panel_age['age_enfant'],
                                  bins=[0, 5, 9, 12, 15, 18, 30],
                                  labels=['0-5', '6-9', '10-12', '13-15', '16-18', '19+'])

taux_age = (panel_age.groupby(['groupe_age', 'wave'])['scolarise']
            .apply(lambda s: (s == 'Oui').mean() * 100)
            .unstack('wave')
            .round(1))
taux_age['Δ (pp)'] = (taux_age[2022] - taux_age[2021]).round(1)
print('Taux de scolarisation par groupe d\'âge × vague (panel) :')
print(taux_age.to_string())

In [ ]:
# 4c. Visualisation : courbe scolarisation × âge × vague
panel_age_fine = panel.dropna(subset=['scolarise', 'age_enfant']).copy()
panel_age_fine = panel_age_fine[panel_age_fine['age_enfant'].between(3, 25)]

courbe = (panel_age_fine.groupby(['age_enfant', 'wave'])['scolarise']
          .apply(lambda s: (s == 'Oui').mean() * 100)
          .unstack('wave'))

fig, ax = plt.subplots(figsize=(10, 5))
courbe.plot(ax=ax, marker='o', lw=2)
ax.set_xlabel('Âge de l\'enfant')
ax.set_ylabel('Taux de scolarisation (%)')
ax.set_title('Taux de scolarisation par âge × vague (enfants suivis aux 2 vagues)')
ax.set_ylim(0, 105)
ax.axhline(50, color='gray', alpha=0.3, lw=1)
ax.legend(title='Vague')
plt.tight_layout()
plt.show()

**Lecture clé** : si les deux courbes sont quasi superposées, la baisse "brute" observée entre 2021 et 2022 vient quasi-entièrement du vieillissement de cohorte. Si la courbe 2022 est nettement en dessous à âge égal, on a un effet net post-COVID. C'est exactement ce qu'on regarde ici.

## 5. Construction de la série temporelle 4 années (2018-19 → 2021-22)

Atout sous-exploité : en 2021, les modules `SC02` et `SC03` posent des questions rétrospectives sur 2019-20 et 2018-19. Combinés avec `SC01` (2020-21) et `cfr02` (2021-22), on a **4 années consécutives** pour chaque enfant du panel.

In [ ]:
# Reconstruction de la série temporelle 4 ans pour les enfants panel
# 2018-19 : classe_2018_19 non-null = scolarisé
# 2019-20 : classe_2019_20 non-null = scolarisé
# 2020-21 : scolarise (vague 2021)
# 2021-22 : scolarise (vague 2022)

p21 = (enfants[(enfants['wave'] == 2021) & (enfants['statut_enfant'] == 'panel_present')]
       [['hh_id', 'enfant_slot', 'age_enfant', 'sexe_enfant',
         'classe_2018_19', 'classe_2019_20', 'scolarise']]
       .rename(columns={'scolarise': 'scolarise_2020_21'}))
p22 = (enfants[(enfants['wave'] == 2022) & (enfants['statut_enfant'] == 'panel_present')]
       [['hh_id', 'enfant_slot', 'scolarise']]
       .rename(columns={'scolarise': 'scolarise_2021_22'}))

p = p21.merge(p22, on=['hh_id', 'enfant_slot'], how='inner')
print(f"Panel reconstitué pour série 4 ans : {len(p)} enfants")
print(p.head(3).to_string())

In [ ]:
# Calcul des taux par année (sans condition d'âge — vue brute)
def taux_2018(s): return s.notna().mean() * 100
def taux_2019(s): return s.notna().mean() * 100
def taux_oui(s): return (s == 'Oui').mean() * 100

serie_brut = pd.Series({
    '2018-19 (pré-COVID)': taux_2018(p['classe_2018_19']),
    '2019-20 (COVID closure)': taux_2019(p['classe_2019_20']),
    '2020-21 (réouverture)': taux_oui(p['scolarise_2020_21']),
    '2021-22 (normalisation)': taux_oui(p['scolarise_2021_22']),
}).round(1)
print('Série brute (non contrôlée pour âge) :')
print(serie_brut.to_string())

**Limite forte** : sans contrôle d'âge, la série monte mécaniquement entre 2018 et 2021 car les enfants entrent dans l'âge scolaire au fil du temps. Pour interpréter, il faut isoler une cohorte d'âge stable (par exemple : enfants qui avaient déjà 6 ans en 2018-19 et n'ont pas dépassé 15 ans en 2021-22, soit ~6-12 ans en 2018-19).

In [ ]:
# Restriction à une cohorte stable : 6-12 ans en 2018-19 (donc 9-15 ans en 2021-22)
# Approximation : age_enfant en 2021 ≈ âge 2020-21, donc âge 2018-19 ≈ age_enfant - 2
p_cohorte = p[(p['age_enfant'] - 2).between(6, 12)].copy()
print(f"Cohorte stable (6-12 ans en 2018-19) : {len(p_cohorte)} enfants")

serie_cohorte = pd.Series({
    '2018-19': taux_2018(p_cohorte['classe_2018_19']),
    '2019-20': taux_2019(p_cohorte['classe_2019_20']),
    '2020-21': taux_oui(p_cohorte['scolarise_2020_21']),
    '2021-22': taux_oui(p_cohorte['scolarise_2021_22']),
}).round(1)
print('\nSérie sur cohorte stable :')
print(serie_cohorte.to_string())

# Visualisation
fig, ax = plt.subplots(figsize=(9, 4.5))
serie_cohorte.plot(kind='line', marker='o', ms=10, lw=2.5, ax=ax, color='steelblue')
ax.set_ylabel('Taux de scolarisation (%)')
ax.set_title('Série de scolarisation 2018-2022 (cohorte stable : 6-12 ans en 2018-19)')
ax.set_ylim(0, 105)
ax.axhline(serie_cohorte.iloc[0], color='gray', ls='--', alpha=0.5, label='Niveau 2018-19')
ax.legend()
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 6. Question analytique 2 — Hétérogénéité par genre

In [ ]:
# Scolarisation par genre × vague, à âge contrôlé
het_genre = (panel.dropna(subset=['scolarise', 'sexe_enfant', 'age_enfant'])
             .groupby(['sexe_enfant', 'wave'])['scolarise']
             .apply(lambda s: (s == 'Oui').mean() * 100)
             .unstack('wave')
             .round(1))
het_genre['Δ (pp)'] = (het_genre[2022] - het_genre[2021]).round(1)
print(het_genre.to_string())

In [ ]:
# Courbe scolarisation × âge × sexe, vague 2022
courbe_sexe = (panel.dropna(subset=['scolarise', 'sexe_enfant', 'age_enfant'])
               .query('age_enfant.between(3, 22)')
               .groupby(['age_enfant', 'sexe_enfant', 'wave'])['scolarise']
               .apply(lambda s: (s == 'Oui').mean() * 100)
               .reset_index(name='taux'))

g = sns.relplot(data=courbe_sexe, x='age_enfant', y='taux', hue='sexe_enfant',
                col='wave', kind='line', marker='o', height=4, aspect=1.2)
g.set_axis_labels('Âge', 'Taux de scolarisation (%)')
g.set_titles('Vague {col_name}')
for ax in g.axes.flat:
    ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

## 7. Question analytique 3 — Transitions individuelles 2021 → 2022

Pour chaque enfant du panel, on observe son statut aux deux vagues. La matrice de transition révèle qui décroche, qui revient, qui reste dehors.

In [ ]:
# Matrice de transition scolarisation
panel_wide = (panel.dropna(subset=['scolarise'])
              .pivot_table(index=['hh_id', 'enfant_slot'],
                           columns='wave', values='scolarise', aggfunc='first')
              .dropna())
panel_wide.columns = ['scol_2021', 'scol_2022']

transitions = pd.crosstab(panel_wide['scol_2021'], panel_wide['scol_2022'],
                          margins=True, margins_name='Total')
print('Matrice de transition (effectifs) :')
print(transitions.to_string())
print()
trans_pct = (pd.crosstab(panel_wide['scol_2021'], panel_wide['scol_2022'],
                          normalize='index') * 100).round(1)
print('Probabilités conditionnelles (P[2022 | 2021]) en % :')
print(trans_pct.to_string())

**Lectures** :
- Diagonale Oui→Oui : enfants en continuité scolaire (cas dominant)
- Oui→Non : **décrocheurs** (cible prioritaire d'intervention)
- Non→Oui : retours à l'école (signal positif de rattrapage)
- Non→Non : enfants durablement hors système

In [ ]:
# Caractériser les décrocheurs (Oui en 2021 → Non en 2022)
decroche = panel_wide.reset_index().query('scol_2021 == "Oui" and scol_2022 == "Non"')
continu = panel_wide.reset_index().query('scol_2021 == "Oui" and scol_2022 == "Oui"')
print(f"Effectif décrocheurs (Oui→Non) : {len(decroche)}")
print(f"Effectif continus (Oui→Oui)    : {len(continu)}")

# Profil démographique (jointure avec données 2021)
ref = (enfants[enfants['wave'] == 2021]
       [['hh_id', 'enfant_slot', 'age_enfant', 'sexe_enfant', 'region', 'niveau_educ_cm']])

dec_full = decroche.merge(ref, on=['hh_id', 'enfant_slot'], how='left')
con_full = continu.merge(ref, on=['hh_id', 'enfant_slot'], how='left')

print(f"\nÂge moyen en 2021 :")
print(f"  Décrocheurs : {dec_full['age_enfant'].mean():.1f} ans (n={dec_full['age_enfant'].notna().sum()})")
print(f"  Continus    : {con_full['age_enfant'].mean():.1f} ans (n={con_full['age_enfant'].notna().sum()})")

print(f"\nPart de filles :")
print(f"  Décrocheurs : {(dec_full['sexe_enfant'] == 'Femme').mean() * 100:.1f}%")
print(f"  Continus    : {(con_full['sexe_enfant'] == 'Femme').mean() * 100:.1f}%")

print(f"\nTop régions des décrocheurs :")
print(dec_full['region'].value_counts().head(5).to_string())

## 8. Question analytique 4 — Motifs de non-scolarisation

In [ ]:
# Top motifs déclarés par vague
mot = (enfants.dropna(subset=['raison_non_scol'])
       .groupby(['wave', 'raison_non_scol'])
       .size()
       .reset_index(name='n')
       .sort_values(['wave', 'n'], ascending=[True, False]))

print('Top motifs par vague :')
for w in [2021, 2022]:
    sub = mot[mot['wave'] == w]
    total = sub['n'].sum()
    sub = sub.assign(pct=(sub['n'] / total * 100).round(1)).head(10)
    print(f"\n--- Vague {w} (total = {total} cas) ---")
    print(sub[['raison_non_scol', 'n', 'pct']].to_string(index=False))

## 9. Synthèse EDA et points d'attention pour `02_analysis.ipynb`

À documenter ici une fois le notebook tourné chez toi sur l'intégralité des données. Quelques hypothèses qu'on cherchera à confirmer/infirmer dans l'analyse :

1. **La "baisse de 10 points" de scolarisation brute entre 2021 et 2022 est-elle réelle ?** À vérifier en isolant l'effet de cohorte (section 4b/4c). Si à âge égal le différentiel disparaît, c'est purement démographique. Si un différentiel persiste, on a un signal post-COVID.

2. **L'inégalité de genre observée : artefact ou réalité ?** Le sens (filles > garçons en scolarisation) est inhabituel pour l'Afrique de l'Ouest et mérite confirmation — possiblement biais d'échantillon (zones rurales vs urbaines, effet projet d'enquête).

3. **Décrochage 2021→2022 : qui ?** L'analyse économétrique en `02_analysis` modélisera la probabilité de décrochage en fonction des caractéristiques observables en 2021 (âge, sexe, classe, type d'école, ménage), avec effets fixes ménage pour les questions intra-ménage.

4. **Motifs : COVID résiduel ou structurel ?** Si les motifs invoqués en 2022 incluent peu de mentions COVID et beaucoup de raisons structurelles (mariage, travail, distance école), la pandémie a cessé d'être le facteur dominant. Si les mentions sanitaires persistent, l'effet est encore vivace.

---

**Prochaine étape** : `02_analysis.ipynb` — modélisation économétrique des transitions, régressions à effets fixes, et production des graphiques finaux destinés au dashboard Power BI et à la réplication R.